In [0]:
%run "../includes/librerias"

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
#Definimos parametros
v_archivo = "movie_language"
v_esquema = "movie_silver"
v_tabla = "movies_languages"
v_partition = "file_date"
v_merge_condition = "target.movie_id = source.movie_id and target.language_id = source.language_id"

#Ingesta de carpeta con archivos JSON

In [0]:
#1. Leer archivos JSON usando DataFrameReader de Spark

# Define el la estructura personName
movie_language_schema = StructType(fields = [
    StructField("movieId", IntegerType(), True),
    StructField("languageId", IntegerType(), True),
    StructField("languageRoleId", IntegerType(), True)
])

# Cargamos el archivo utilizando la estructura definida
movie_language_df = spark.read\
    .schema(movie_language_schema)\
    .option("multiLine", "true")\
    .json(f"{bronze_folder_path}/{v_file_date}/{v_archivo}")

# Mostramos el resultado
display(movie_language_df.filter(col("movieId") == 5))

In [0]:
#Paso 2 - Renombrar, añadir y dar formato a las columnas requeridas

movie_language_renamed_df = movie_language_df\
    .withColumnRenamed("movieId", "movie_id")\
    .withColumnRenamed("languageId", "language_id")

movie_language_renamed_df = add_ingestion_date(movie_language_renamed_df)
movie_language_renamed_df = add_env(movie_language_renamed_df)   
movie_language_renamed_df = add_file_date (movie_language_renamed_df)

display(movie_language_renamed_df.limit(10))


In [0]:
#Paso 3 - Seleccionar las columnas que se requieren 

final_df = movie_language_renamed_df.drop(col("languageRoleId") )
                                                   
display(final_df.limit(10))



In [0]:
#Paso 5 - borramos la informacion de la tabla antes de cargarla
resultado = merge_delta_lake (v_esquema, v_tabla, final_df, v_merge_condition, v_partition)
print(resultado)

In [0]:
#Paso 4 - Guardar datos en datalake en formato parket 
#movie_language_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.movies_languages")

#movie_language_df.write.mode("append").partitionBy(v_partition).format("delta").saveAsTable(f"{v_esquema}.{v_tabla}")
#print(f"Se insertaron {movie_language_df.count()} registros en la tabla {v_esquema}.{v_tabla}")

In [0]:
dbutils.notebook.exit("El notebook 11. Ingestion folder_movie_language, termino correctamente")